# AIM AI Capstone: End-to-End Reproducible Analysis

This notebook reproduces the core pipeline for the capstone project.

**Important:** The outcome is *recognized developmental concern or support*. It is not a diagnosis and should not be interpreted as latent developmental need.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    TARGET, RANDOM_STATE, MODEL_A_FEATURES, MODEL_B_FEATURES, MODEL_C_FEATURES
)
from src.data import load_nsch, restrict_age_3_to_5, build_target
from src.modeling import make_gb_pipeline
from src.evaluation import cv_summary, out_of_fold_probabilities, threshold_table
from src.fairness import subgroup_metrics

## 2. Load data and construct the cohort

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "raw" / "nsch_2024e_topical.dta"

raw = load_nsch(DATA_PATH)
cohort = restrict_age_3_to_5(raw)
cohort = build_target(cohort)

print("Raw shape:", raw.shape)
print("Ages 3-5 shape:", cohort.shape)
print(cohort[TARGET].value_counts(dropna=False))
print("Positive rate:", cohort[TARGET].mean())

Expected capstone cohort:
- N = 7,485
- Positive = 2,030 (27.1%)
- Negative = 5,455 (72.9%)

## 3. Data quality checks

In [ ]:
model_a = cohort[MODEL_A_FEATURES + [TARGET]].copy()

print("Exact duplicate rows:", model_a.duplicated().sum())

missing_pct = model_a[MODEL_A_FEATURES].isna().mean().mul(100).sort_values(ascending=False)
print("Maximum missingness (%):", missing_pct.max())
display(missing_pct.head(15))

## 4. Train/test split

In [ ]:
X = cohort[MODEL_A_FEATURES].copy()
y = cohort[TARGET].astype(int).copy()

stratify_group = X["sc_age_years"].astype(str) + "_" + y.astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=stratify_group
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

## 5. Final apples-to-apples comparison

In [ ]:
feature_sets = {
    "Model A: Full 102 Features": MODEL_A_FEATURES,
    "Model B: Domain-Informed 38": MODEL_B_FEATURES,
    "Model C: Data-Driven 44": MODEL_C_FEATURES,
}

rows = []
for name, features in feature_sets.items():
    print(f"Evaluating {name}...")
    pipe = make_gb_pipeline(features)
    result = cv_summary(pipe, X_train[features], y_train, folds=5, random_state=RANDOM_STATE)
    result["Model"] = name
    result["Features"] = len(features)
    rows.append(result)

comparison = pd.DataFrame(rows)[
    ["Model","Features","ROC-AUC","PR-AUC","F1","Precision","Sensitivity","Accuracy"]
]
display(comparison)

Reference results from the completed capstone run:

| Model | Features | ROC-AUC | PR-AUC | F1 | Precision | Sensitivity | Accuracy |
|---|---:|---:|---:|---:|---:|---:|---:|
| Model A | 102 | 0.782 | 0.662 | 0.510 | 0.772 | 0.381 | 0.802 |
| Model B | 38 | 0.766 | 0.643 | 0.500 | 0.759 | 0.373 | 0.798 |
| Model C | 44 | 0.785 | 0.664 | 0.518 | 0.770 | 0.390 | 0.803 |

Small numerical differences may occur across library versions.

## 6. Threshold analysis for Model B

In [ ]:
pipe_b = make_gb_pipeline(MODEL_B_FEATURES)
oof_b = out_of_fold_probabilities(
    pipe_b, X_train[MODEL_B_FEATURES], y_train,
    folds=5, random_state=RANDOM_STATE
)

thresholds_b = threshold_table(y_train, oof_b)
display(thresholds_b)

## 7. Fairness audit by sex and age at threshold 0.30

In [ ]:
sex_audit = subgroup_metrics(
    y_train, oof_b, X_train["sc_sex"], threshold=0.30
)
age_audit = subgroup_metrics(
    y_train, oof_b, X_train["sc_age_years"], threshold=0.30
)

print("Sex audit")
display(sex_audit)

print("Age audit")
display(age_audit)

The completed capstone analysis observed lower sensitivity among females than males at a common 0.30 threshold.

This must be interpreted cautiously because the outcome measures **recognized concern/support**, not latent developmental need. Historical differences in recognition and service access may therefore be embedded in the labels themselves.

## 8. Limitations

- U.S. survey data; no Philippine external validation
- outcome is recognized concern/support, not diagnosis or latent need
- threshold has not been clinically validated
- Model C feature selection was exploratory rather than fully nested
- subgroup disparities require explicit mitigation before any future deployment
- survey weights were not incorporated into the predictive-model training workflow